# League of Legends — Player Churn and Retention
## Notebook 01 — Data Collection

**Business question.** Among ranked players who have already invested real time,
what predicts that they keep playing or go dormant? This is the player retention
question a game analytics team cares about most, because committed players are the
ones a studio most wants to keep.

This notebook documents **how the data was collected**. The heavy API pull is long,
rate limited, and resumable, so it lives in importable modules under `src/`. Here we
explain the design and summarize the collected population. Cleaning, feature
engineering, and analysis follow in notebooks 02 to 04.

## 1. Data source

- **API.** Riot Games API, original data pulled directly, not a public CSV.
- **Region.** EUW1. **Queue.** ranked solo, `RANKED_SOLO_5x5`, match queue id 420.
- **Player keys.** The `league-exp-v4` endpoint returns the stable `puuid` directly
  inside each ranked entry, so the whole pipeline keys on `puuid` and skips the older
  `summoner-v4` conversion step.
- **Seed.** A stratified sample across every tier from Iron to Challenger, an equal
  target per tier, so the data carries real variation in skill and behavior. Tier is
  kept as a feature.

## 2. Observation window and churn definition

Each player is observed over a six month window, split into two periods:

- **Feature period**, the first four months. All behavioral features are measured here.
- **Outcome period**, the last two months. Used only to label churn.

**Engaged player.** Included only if they played at least 50 ranked games in the
feature period. This focuses the study on players who invested real time, not casual
players who drift off.

**Churn.** A player is churned if they played zero ranked games in the outcome period.
The label needs only a match count, so no match detail is pulled for the outcome
period, which saves a large amount of API work.

In [1]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("..") / "data"
pd.set_option("display.max_columns", None)

In [2]:
collection = {
    "region": "EUW1",
    "queue": "RANKED_SOLO_5x5 (id 420)",
    "observation_window_months": 6,
    "feature_period_months": 4,
    "outcome_period_months": 2,
    "min_feature_games": 50,
    "churn_rule": "zero ranked games in the outcome period",
    "seed_strategy": "stratified across all tiers, Iron to Challenger",
}
for k, v in collection.items():
    print(f"{k:28} {v}")

region                       EUW1
queue                        RANKED_SOLO_5x5 (id 420)
observation_window_months    6
feature_period_months        4
outcome_period_months        2
min_feature_games            50
churn_rule                   zero ranked games in the outcome period
seed_strategy                stratified across all tiers, Iron to Challenger


## 3. The pull pipeline (`src/`)

The collection runs in three restartable modules:

- **`seed_players.py`** pulls the ladder with `league-exp-v4` and builds a stratified
  player list, reading the `puuid` directly from each ranked entry.
- **`build_population.py`** pulls ranked match ids separately for the feature and
  outcome periods, keeps only engaged players, labels churn from the outcome count,
  and writes `population.json`. Already processed players are skipped, so the run
  resumes after an interruption.
- **`pull_match_details.py`** pulls full match json for the feature period only, and
  for churned players first, so the rare class is captured early. Each match is cached
  to disk and skipped if already present.

The shared client `riot_client.py` handles the development key limits with backoff on
429, retries on server errors, and a clear message on an expired key so a multi day
pull can refresh the key and resume.

## 4. The collected population

The summary reads `population.json`, the direct output of the collection step. It
holds every seeded player with an engaged flag, and for engaged players the churn
label derived from the outcome period.

In [3]:
pop = pd.DataFrame(json.load(open(DATA / "population.json", encoding="utf-8")))

processed = len(pop)
engaged = pop[pop["engaged"]]
churned = int(engaged["churned"].sum())

print(f"Players seeded and processed : {processed}")
print(f"Engaged (>= 50 feature games): {len(engaged)}")
print(f"Filtered out as not engaged  : {processed - len(engaged)}")
print(f"  of engaged -> churned      : {churned} ({100 * churned / len(engaged):.1f}%)")
print(f"  of engaged -> retained     : {len(engaged) - churned}")

Players seeded and processed : 1000
Engaged (>= 50 feature games): 610
Filtered out as not engaged  : 390
  of engaged -> churned      : 48 (7.9%)
  of engaged -> retained     : 562


In [4]:
order = ["IRON", "BRONZE", "SILVER", "GOLD", "PLATINUM",
         "EMERALD", "DIAMOND", "MASTER", "GRANDMASTER", "CHALLENGER"]

# cast to plain integers so the rates compute on any pandas version
pop["engaged_n"] = pop["engaged"].astype(int)
pop["churned_n"] = pop["churned"].fillna(False).astype(int)

by_tier = pop.groupby("tier").agg(
    processed=("puuid", "size"),
    engaged=("engaged_n", "sum"),
    churned=("churned_n", "sum"),
)
by_tier["engagement_rate_%"] = (100 * by_tier["engaged"] / by_tier["processed"]).round(1)
by_tier["churn_rate_%"] = (100 * by_tier["churned"] / by_tier["engaged"]).round(1)
by_tier.reindex(order)

C:\Users\talja\AppData\Local\Temp\ipykernel_10960\98643472.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pop["churned_n"] = pop["churned"].fillna(False).astype(int)


,processed,engaged,churned,engagement_rate_%,churn_rate_%
tier,,,,,
IRON,100,21,5,21.0,23.8
BRONZE,100,30,4,30.0,13.3
SILVER,100,39,6,39.0,15.4
GOLD,100,41,7,41.0,17.1
PLATINUM,100,51,4,51.0,7.8
EMERALD,100,76,22,76.0,28.9
DIAMOND,100,88,0,88.0,0.0
MASTER,100,87,0,87.0,0.0
GRANDMASTER,100,89,0,89.0,0.0


**Read.** Engagement rises sharply with tier. Few low tier players reach 50 ranked
games in four months, while almost all high tier players do. Churn runs the other way,
it sits in the low and mid tiers and disappears in the high tiers. Notebook 04 returns
to this and studies the behavioral drivers where churn actually varies.

**Next:** `02_parsing_and_cleaning.ipynb` turns the cached match json into a tidy
player match table.